# 第 2 周第 1 天 —— 四向自闭症对话（Steph Moyo）

该笔记本遵循第 2 周第 1 天的多角色对话模式：**每个代理每次都会收到完整的对话历史**。

## 参加者

| 角色 | 模型 | 客户端 |
|------|------|--------|
| 医生 Dr. Amina | `gpt-5-nano` | OpenAI |
| 心理学家 Dr. Leo | Ollama `llama3.2` | 本地 OpenAI 兼容接口 |
| 儿科医生 Dr. Nia | `gpt-4.1-mini` | OpenAI |
| 测试对象 Sam（儿童角色） | `gpt-4o-mini` | OpenAI |

> 仅限教育模拟。不是医疗诊断或治疗计划。

## 怎么跑

1. `.env` 配置 `OPENAI_API_KEY`；本机 Ollama 已启动并拉取 `llama3.2`
2. 依次运行：初始化 → 定义角色 → 定义回合函数 → 种子对话 → 多轮循环


In [ ]:
# ========== 环境 + 双客户端：云端 OpenAI 与本地 Ollama（OpenAI 兼容 /v1） ==========

# 标准库 os：读环境变量
import os
# 从 dotenv 导入 load_dotenv：把 .env 读进环境
from dotenv import load_dotenv
# 从 openai 导入 OpenAI：云端与 Ollama 都用同一套 SDK 接口
from openai import OpenAI
# 从 IPython.display 导入 Markdown / display：在笔记本里漂亮展示每轮发言
from IPython.display import Markdown, display

# 加载 .env；override=True 允许覆盖已有环境变量
load_dotenv(override=True)
# 读取 OPENAI_API_KEY（名字必须保持）
openai_api_key = os.getenv("OPENAI_API_KEY")

# 有密钥则打印前 8 位做存在性确认；提示字符串保持英文
if openai_api_key:
    print(f"OpenAI API key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API key is missing. Add OPENAI_API_KEY to your .env file.")

# 默认云端客户端（读环境变量密钥）
openai = OpenAI()
# 本地 Ollama：OpenAI 兼容基址；api_key 占位字符串 "ollama" 保持原样
ollama = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")


In [ ]:
# ========== 四位参加者：每人含 name / model / client / system（system 英文勿改） ==========

participants = [
    {
        # 全科医生角色：用 gpt-5-nano，走 openai 客户端
        "name": "Dr. Amina (General Doctor)",
        "model": "gpt-5-nano",
        "client": "openai",
        "system": """You are a licensed general doctor.
Speak clearly for parents of autistic children.
Use evidence-based recommendations, stay compassionate, and avoid stigma.
Do not prescribe medications in detail; suggest professional follow-up when needed.""",
    },
    {
        # 儿童心理医生：本地 llama3.2，走 ollama 客户端
        "name": "Dr. Leo (Child Psychologist)",
        "model": "llama3.2",
        "client": "ollama",
        "system": """You are a child psychologist using trauma-informed, neurodiversity-affirming language.
Focus on emotional regulation, communication, routine support, and family coaching.""",
    },
    {
        # 儿科医生：gpt-4.1-mini
        "name": "Dr. Nia (Pediatric Doctor)",
        "model": "gpt-4.1-mini",
        "client": "openai",
        "system": """You are a pediatric doctor in the GPT-4 family.
Prioritize early developmental support, referrals, and practical parent guidance.""",
    },
    {
        # 5 岁儿童人设：仅模拟，不给医疗建议
        "name": "Sam (Test Subject, 5-year-old child persona)",
        "model": "gpt-4o-mini",
        "client": "openai",
        "system": """You are role-playing a 5-year-old autistic child for simulation only.
Use very short, simple sentences and describe feelings/sensory experiences.
Do not give medical advice.""",
    },
]


In [ ]:
# ========== 对话工具：格式化历史、按客户端调模型、组织单回合 user prompt ==========

def format_history(history):
    # 把 [{speaker, message}, ...] 打成多行「说话人: 内容」文本，供拼进 prompt
    return "\n".join([f"{turn['speaker']}: {turn['message']}" for turn in history])


def call_model(client_name, model, messages):
    # 按 client 名字选择云端 openai 或本地 ollama
    client = openai if client_name == "openai" else ollama
    # 额外 kwargs：部分 gpt-5* 模型需要 reasoning_effort
    kwargs = {}
    if model.startswith("gpt-5"):
        # 降低推理开销；参数名与取值保持原样
        kwargs["reasoning_effort"] = "minimal"
    # 统一走 chat.completions.create；**kwargs 展开可选参数
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        **kwargs
    )
    # 取助手正文并去掉首尾空白
    return response.choices[0].message.content.strip()


def one_turn(participant, history):
    # 当前完整对话历史（含家长开场 + 已有各角色回复）
    conversation_so_far = format_history(history)
    # user prompt：指定以谁的身份回答 + 主题 + 历史 + 回复规则（英文保留）
    user_prompt = f"""
You are now responding as {participant['name']}.

Topic: autism in a 5-year-old child and the best supports/treatments.

Conversation so far:
{conversation_so_far}

Response rules:
- Keep your answer to 3-5 sentences.
- Be non-stigmatizing and practical.
- Prefer evidence-based supports like early intervention, speech therapy, occupational therapy, parent coaching, and school accommodations.
- Mention that treatment should be personalized.
"""

    # system 用该角色人设；user 用上面拼好的回合指令
    messages = [
        {"role": "system", "content": participant["system"]},
        {"role": "user", "content": user_prompt},
    ]

    # 按该角色的 client / model 真正发请求
    return call_model(participant["client"], participant["model"], messages)


In [ ]:
# ========== 种子对话：家长开场问题，并先展示出来 ==========

# 对话历史列表：每项含 speaker 与 message
conversation_history = [
    {
        "speaker": "Parent",
        # 家长问题英文保留（会进入后续每轮的 history prompt）
        "message": "My child is 5 years old with autism. What are the best treatments to help communication, behavior, and sleep?"
    }
]

# 用 Markdown 标题展示家长第一句
display(Markdown(f"### {conversation_history[0]['speaker']}\n{conversation_history[0]['message']}"))


In [ ]:
# ========== 多轮循环：rounds 轮 × 四位参加者，每人读全历史后发言 ==========

# 外层轮数
rounds = 3

# 跑 rounds 轮；每轮按 participants 顺序轮流发言
for _ in range(rounds):
    for participant in participants:
        # 基于当前完整 history 生成该角色回复
        reply = one_turn(participant, conversation_history)
        # 把新发言追加进历史，供下一位/下一轮使用
        conversation_history.append({"speaker": participant["name"], "message": reply})
        # 展示：角色名 + 所用模型 + 回复正文
        display(Markdown(f"### {participant['name']} ({participant['model']})\n{reply}"))

# 结束提示字符串保持英文原样
print("Conversation complete.")


In [ ]:
# ========== 可选：打印完整逐字稿（speaker: message） ==========

# 遍历 conversation_history，用纯文本再看一遍整场对话
for turn in conversation_history:
    print(f"{turn['speaker']}: {turn['message']}\n")
